# Training Deficits & Risk ClassificationConverted from `src/training_deficits.py`---

**Beschreibung:** Training Deficits - Identifies training needs and disciplinary actions

In [1]:
import pandas as pdimport numpy as npfrom pathlib import Path# Thresholds for classificationTHRESHOLDS = {    # YELLOW (Training recommended)    'low_score': 2.5,            # Avg score below 2.5    'high_variance': 1.5,        # Std > 1.5 (inconsistent)    'below_team_avg': -0.5,      # 0.5 points below team avg    # RED (Disciplinary)    'critical_low_score': 1.5,   # Avg score below 1.5    'critical_min_score': 2.0,   # Min score below 2}def calculate_employee_metrics(scored_df):    """    Calculate performance metrics per employee.    Returns:        DataFrame with aggregated metrics    """    print(" Calculating employee metrics...")    # Only valid scores    valid_df = scored_df[scored_df['Q1'] > 0].copy()    # Aggregation per assignee    metrics = valid_df.groupby('assignee').agg({        'Q1': ['mean', 'std', 'count', 'min'],        'Q2': ['mean'],        'Q3': ['mean']    }).reset_index()    # Rename columns    metrics.columns = [        'employee_id',        'avg_q1', 'std_q1', 'ticket_count', 'min_q1',        'avg_q2', 'avg_q3'    ]    # Q1 and Q2 both measure "Quality of Work", Q3 measures "Client Relations"    # Therefore: Quality Score = (Q1 + Q2) / 2, then 50% Quality + 50% Client    metrics['quality_score'] = (metrics['avg_q1'] + metrics['avg_q2']) / 2    metrics['client_score'] = metrics['avg_q3']    # Overall score: 50% Quality of Work + 50% Client Relations    metrics['overall_score'] = (        metrics['quality_score'] * 0.5 +        metrics['client_score'] * 0.5    )    # Comparison with team average    team_avg = metrics['overall_score'].mean()    metrics['vs_team_avg'] = metrics['overall_score'] - team_avg    print(f"   {len(metrics)} employees analyzed")    print(f"   Team average: {team_avg:.2f}")    return metricsdef classify_employee(metrics_row):    """    Classify an employee by risk level.    Returns:        dict: Classification and recommendations    """    training_areas = []    disciplinary_flags = []    recommendations = []    avg_score = metrics_row['overall_score']    std_score = metrics_row['std_q1'] if pd.notna(metrics_row['std_q1']) else 0    min_score = metrics_row['min_q1']    vs_team = metrics_row['vs_team_avg']    # === TRAINING (YELLOW) ===    if avg_score < THRESHOLDS['low_score']:        training_areas.append('Solution Quality')        recommendations.append('Workshop: Systematic Problem Analysis')    if std_score > THRESHOLDS['high_variance']:        training_areas.append('Consistency')        recommendations.append('Coaching: Checklists for consistent quality')    if vs_team < THRESHOLDS['below_team_avg']:        training_areas.append('General Performance')        recommendations.append('Mentoring: Pair work with experienced colleague')    # === DISCIPLINARY (RED) ===    if avg_score < THRESHOLDS['critical_low_score']:        disciplinary_flags.append('Critically low performance')    if min_score < THRESHOLDS['critical_min_score']:        disciplinary_flags.append('Very poor individual ratings')    # === RISK LEVEL ===    if disciplinary_flags:        risk_level = 'RED'    elif training_areas:        risk_level = 'YELLOW'    else:        risk_level = 'GREEN'        recommendations.append('Good work! Keep it up.')    return {        'risk_level': risk_level,        'training_areas': training_areas,        'disciplinary_flags': disciplinary_flags,        'recommendations': recommendations    }def analyze_all_employees(scored_df):    """    Analyze all employees.    Returns:        DataFrame with classification    """    print("\n TRAINING DEFICIT ANALYSIS")    print("="*50)    # Calculate metrics    metrics_df = calculate_employee_metrics(scored_df)    # Classify each employee    results = []    for _, row in metrics_df.iterrows():        classification = classify_employee(row)        results.append({            'employee': row['employee_id'],            'overall_score': round(row['overall_score'], 2),            'ticket_count': int(row['ticket_count']),            'risk_level': classification['risk_level'],            'training_areas': ', '.join(classification['training_areas']) or '-',            'flags': ', '.join(classification['disciplinary_flags']) or '-',            'recommendations': '; '.join(classification['recommendations'])        })    results_df = pd.DataFrame(results)    results_df = results_df.sort_values(['risk_level', 'overall_score'], ascending=[False, True])    # Summary    green = (results_df['risk_level'] == 'GREEN').sum()    yellow = (results_df['risk_level'] == 'YELLOW').sum()    red = (results_df['risk_level'] == 'RED').sum()    print(f"\n RESULT:")    print(f"    GREEN (OK): {green}")    print(f"    YELLOW (Training): {yellow}")    print(f"    RED (Disciplinary): {red}")    return results_dfdef print_training_plan(results_df):    """Print a training plan."""    print("\n" + "="*50)    print(" TRAINING PLAN")    print("="*50)    # RED employees    red_employees = results_df[results_df['risk_level'] == 'RED']    if len(red_employees) > 0:        print("\n URGENT - Immediate action required:")        for _, emp in red_employees.iterrows():            print(f"\n   {emp['employee']} (Score: {emp['overall_score']})")            print(f"      Flags: {emp['flags']}")    # YELLOW employees    yellow_employees = results_df[results_df['risk_level'] == 'YELLOW']    if len(yellow_employees) > 0:        print("\n TRAINING RECOMMENDED:")        for _, emp in yellow_employees.head(10).iterrows():            print(f"\n   {emp['employee']} (Score: {emp['overall_score']})")            print(f"      Areas: {emp['training_areas']}")    if len(red_employees) == 0 and len(yellow_employees) == 0:        print("\n All employees in green zone!")

##  Execution

In [2]:
print("="*50)print(" TRAINING DEFICITS")print("="*50)# Load datadata_path = Path("data/raw/issues_snapshot_sample.xlsx")if data_path.exists():    scored_df = pd.read_excel(data_path)    print(f" Loaded: {len(scored_df)} rated samples")    # Analysis    results_df = analyze_all_employees(scored_df)    # Training plan    print_training_plan(results_df)    # Save    output_path = Path("reports/training_report.csv")    output_path.parent.mkdir(parents=True, exist_ok=True)    results_df.to_csv(output_path, index=False)    print(f"\n Saved: {output_path}")else:    print(" Rated samples not found!")

================================================== TRAINING DEFICITS================================================== Rated samples not found!